# Pebble voice — frozen speech-encoder backbone selection

**emotion2vec (primary) vs WavLM-Large (baseline)** for crisis-sensitive speech affect.
Pebble's heterogeneous heads (emotion softmax + a high-distress recall head) on RAVDESS, 3 seeds, paired delta. Mirrors `pebble-mlm-3seed`. Ref: `docs/voice-method-selection.md`.

## 0. Install pinned audio stack  (run once)

In [ ]:
# Cell 0 — pinned audio stack (P100 = sm_60; Kaggle default torch drops it -> crash).
import os
os.environ["PYTHONIOENCODING"] = "utf-8"
os.environ["PYTHONUTF8"] = "1"
# torchvision pinned to match torch 2.5.1 — else transformers' lazy torchvision import
# fails with "operator torchvision::nms does not exist" and WavLM won't load.
get_ipython().system('pip install -q torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 --index-url https://download.pytorch.org/whl/cu121')
get_ipython().system('pip install -q transformers==4.48.2 datasets==3.2.0 librosa soundfile scikit-learn scipy')
# emotion2vec route (primary backbone). Heavy dep tree; usage is guarded so a
# failed install/import only skips the emotion2vec arm — WavLM still produces a result.
get_ipython().system('pip install -q funasr modelscope || echo "funasr install failed -> emotion2vec arm will be skipped"')
print("install cell done")


## 1. Imports & config

In [ ]:
# Cell 1 — imports & config.
# Experiment: FROZEN speech-encoder backbone selection for crisis-sensitive affect.
#   emotion2vec (primary) vs WavLM-Large (baseline), Pebble's heterogeneous heads
#   (emotion softmax + a high-distress recall head), RAVDESS, 3 seeds, paired delta.
# Mirrors the repo's pebble-mlm-3seed methodology: mean +/- std + per-seed delta.
import os, json, random, warnings, tempfile
warnings.filterwarnings("ignore")
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import f1_score, recall_score, precision_score

print("torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ART = "/kaggle/working"
SR = 16000
MAX_SEC = 4.0
MAX_SAMPLES = int(SR * MAX_SEC)
SEEDS = [13, 42, 1337]                 # matches repo config.py
TRAIN_ACTORS = set(range(1, 21))       # speaker-independent split (actors 1-20 train)
VAL_ACTORS = set(range(21, 25))        # actors 21-24 val
PROBE_EPOCHS, PROBE_LR, HEAD_DIM = 60, 1e-3, 256
RECALL_FLOOR = 0.90                    # Pebble's safety-head recall-floor framing (voice proxy)
SAFETY_POS_WEIGHT = 3.0               # up-weight the distress positive class (high-recall)

EMOTIONS = ["neutral", "calm", "happy", "sad", "angry", "fearful", "disgust", "surprised"]
EMO2ID = {e: i for i, e in enumerate(EMOTIONS)}
N_EMO = len(EMOTIONS)
# Distress = the high-arousal / negative-valence emotions -> the safety-head analogue.
DISTRESS = {"sad", "angry", "fearful", "disgust"}

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

print(f"emotions={EMOTIONS}\ndistress(positive)={sorted(DISTRESS)} | seeds={SEEDS}")


## 2. RAVDESS data  (resample 48k->16k, speaker-independent split)

In [ ]:
# Cell 2 — RAVDESS load, resample to 16 kHz, speaker-independent split.
import torchaudio
from datasets import load_dataset

ds = load_dataset("narad/ravdess", split="train", trust_remote_code=True)
print("RAVDESS:", ds)
INT2STR = ds.features["labels"].int2str

def fix_len(wav):
    if len(wav) >= MAX_SAMPLES:
        return wav[:MAX_SAMPLES]
    return np.pad(wav, (0, MAX_SAMPLES - len(wav)))

def parse_actor(ex):
    # speaker_id like "Actor_07" or an int; be robust.
    sid = ex.get("speaker_id")
    if isinstance(sid, str):
        digits = "".join(c for c in sid if c.isdigit())
        return int(digits) if digits else -1
    return int(sid) if sid is not None else -1

def to_records(ds):
    train, val = [], []
    for ex in ds:
        a = ex["audio"]
        wav = np.asarray(a["array"], dtype=np.float32)
        src_sr = a["sampling_rate"]
        if src_sr != SR:
            wav = torchaudio.functional.resample(torch.from_numpy(wav), src_sr, SR).numpy()
        wav = fix_len(wav.astype(np.float32))
        emo = INT2STR(ex["labels"])
        rec = {"wav": wav, "emo": EMO2ID[emo], "distress": int(emo in DISTRESS)}
        (train if parse_actor(ex) in TRAIN_ACTORS else val).append(rec)
    return train, val

train_recs, val_recs = to_records(ds)
print(f"train={len(train_recs)} val={len(val_recs)} clips "
      f"| train distress+={sum(r['distress'] for r in train_recs)} "
      f"val distress+={sum(r['distress'] for r in val_recs)}")

# stash one val clip as the FastAPI sample (16 kHz mono wav).
import soundfile as sf
sample = val_recs[0]
sf.write(f"{ART}/sample_val.wav", sample["wav"], SR)
print(f"sample wav -> {ART}/sample_val.wav (emo={EMOTIONS[sample['emo']]}, distress={sample['distress']})")


## 3. Frozen feature extraction  (each encoder runs once)

In [ ]:
# Cell 3 — frozen feature extraction (each encoder runs ONCE; heads train on cached embeddings).
from transformers import AutoFeatureExtractor, WavLMModel

def wavlm_extractor():
    fe = AutoFeatureExtractor.from_pretrained("microsoft/wavlm-large")
    model = WavLMModel.from_pretrained("microsoft/wavlm-large").to(DEVICE).eval()
    @torch.no_grad()
    def extract(wavs, bs=16):
        out = []
        for i in range(0, len(wavs), bs):
            batch = [w for w in wavs[i:i + bs]]
            enc = fe(batch, sampling_rate=SR, return_tensors="pt", padding=True)
            enc = {k: v.to(DEVICE) for k, v in enc.items()}
            with torch.cuda.amp.autocast(enabled=DEVICE.type == "cuda"):
                h = model(**enc).last_hidden_state            # (B, T, 1024)
            out.append(h.float().mean(dim=1).cpu().numpy())   # mean-pool over time
        return np.concatenate(out, 0)
    return extract, 1024

def emotion2vec_extractor():
    # Primary backbone. Guarded: any failure here -> caller skips this arm.
    # "model not registered" is FunASR masking a modelscope-CN download failure;
    # hub="hf" pulls from HuggingFace instead, with a snapshot_download fallback.
    from funasr import AutoModel as FunASR
    try:
        model = FunASR(model="iic/emotion2vec_base", hub="hf", disable_update=True)
    except Exception:
        from huggingface_hub import snapshot_download
        local = snapshot_download("emotion2vec/emotion2vec_base")
        model = FunASR(model=local, disable_update=True)
    def extract(wavs, bs=None):
        feats = []
        for w in wavs:
            with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as tf:
                sf.write(tf.name, w, SR); path = tf.name
            r = model.generate(path, granularity="utterance", extract_embedding=True,
                               output_dir=None)
            feats.append(np.asarray(r[0]["feats"], dtype=np.float32))
            os.remove(path)
        return np.stack(feats)                                 # (N, 768)
    return extract, 768

import soundfile as sf
BACKBONES = {}   # name -> {"train": arr, "val": arr, "dim": int}
train_wavs = [r["wav"] for r in train_recs]
val_wavs = [r["wav"] for r in val_recs]

for name, builder in [("emotion2vec", emotion2vec_extractor), ("wavlm-large", wavlm_extractor)]:
    try:
        print(f"\n[{name}] building extractor...")
        extract, dim = builder()
        tr = extract(train_wavs); va = extract(val_wavs)
        BACKBONES[name] = {"train": tr, "val": va, "dim": dim}
        print(f"[{name}] features: train{tr.shape} val{va.shape}")
        del extract; torch.cuda.empty_cache()
    except Exception as e:
        print(f"[{name}] SKIPPED -> {type(e).__name__}: {e}")

assert BACKBONES, "no backbone produced features"
print("\nbackbones with features:", list(BACKBONES))


## 4. Multi-task probe  (emotion softmax + distress BCE, 3 seeds, recall floor)

In [ ]:
# Cell 4 — multi-task probe: emotion softmax + distress BCE, 3 seeds, recall-floored threshold.
# Head architectures mirror src/pebble_llm/models/heads.py so the FastAPI app can reuse them.
class EmotionHead(nn.Module):
    def __init__(self, dim, n, head_dim=HEAD_DIM, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(nn.Dropout(dropout), nn.Linear(dim, head_dim), nn.GELU(),
                                 nn.Dropout(dropout), nn.Linear(head_dim, n))
    def forward(self, x): return self.net(x)

class SafetyHead(nn.Module):
    def __init__(self, dim, head_dim=64, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(nn.Dropout(dropout), nn.Linear(dim, head_dim), nn.GELU(),
                                 nn.Linear(head_dim, 1))
    def forward(self, x): return self.net(x).squeeze(-1)

emo_tr = torch.tensor([r["emo"] for r in train_recs], dtype=torch.long)
dis_tr = torch.tensor([r["distress"] for r in train_recs], dtype=torch.float)
emo_va = np.array([r["emo"] for r in val_recs])
dis_va = np.array([r["distress"] for r in val_recs])

def threshold_at_recall(y, prob, floor):
    # lowest threshold whose recall >= floor; report precision there (Pebble safety framing).
    best_t, best_p = 0.5, 0.0
    for t in np.linspace(0.01, 0.99, 99):
        pred = (prob >= t).astype(int)
        if recall_score(y, pred, zero_division=0) >= floor:
            p = precision_score(y, pred, zero_division=0)
            if t > best_t or best_p == 0.0:
                best_t, best_p = t, p
    return best_t, best_p

def train_probe(Xtr, Xva, dim, seed):
    set_seed(seed)
    Xtr_t = torch.tensor(Xtr, dtype=torch.float, device=DEVICE)
    Xva_t = torch.tensor(Xva, dtype=torch.float, device=DEVICE)
    emo_h, saf_h = EmotionHead(dim, N_EMO).to(DEVICE), SafetyHead(dim).to(DEVICE)
    opt = torch.optim.AdamW(list(emo_h.parameters()) + list(saf_h.parameters()),
                            lr=PROBE_LR, weight_decay=1e-2)
    et, dt = emo_tr.to(DEVICE), dis_tr.to(DEVICE)
    pw = torch.tensor(SAFETY_POS_WEIGHT, device=DEVICE)
    for _ in range(PROBE_EPOCHS):
        emo_h.train(); saf_h.train(); opt.zero_grad()
        loss = (F.cross_entropy(emo_h(Xtr_t), et)
                + F.binary_cross_entropy_with_logits(saf_h(Xtr_t), dt, pos_weight=pw))
        loss.backward(); opt.step()
    emo_h.eval(); saf_h.eval()
    with torch.no_grad():
        emo_pred = emo_h(Xva_t).argmax(-1).cpu().numpy()
        dis_prob = torch.sigmoid(saf_h(Xva_t)).cpu().numpy()
    f1 = f1_score(emo_va, emo_pred, average="macro")
    thr, prec = threshold_at_recall(dis_va, dis_prob, RECALL_FLOOR)
    auroc_rec = recall_score(dis_va, (dis_prob >= 0.5).astype(int), zero_division=0)
    return {"emo_macroF1": f1, "distress_recall@0.5": auroc_rec,
            "thr@floor": thr, "precision@floor": prec}, (emo_h, saf_h, thr)

results = {}     # name -> list of per-seed metric dicts
artifacts = {}   # name -> (emo_h, saf_h, thr) from seed 0 (for serving)
for name, b in BACKBONES.items():
    results[name] = []
    for si, seed in enumerate(SEEDS):
        m, art = train_probe(b["train"], b["val"], b["dim"], seed)
        results[name].append(m)
        if si == 0:
            artifacts[name] = art
        print(f"[{name} seed={seed}] emo_macroF1={m['emo_macroF1']:.4f} "
              f"distress_recall={m['distress_recall@0.5']:.3f} "
              f"prec@recall{RECALL_FLOOR}={m['precision@floor']:.3f} (thr={m['thr@floor']:.2f})")


## 5. Results + serving artifacts

In [ ]:
# Cell 5 — aggregate (mean +/- std), paired delta, and save serving artifacts.
import pandas as pd
HF_ID = {"wavlm-large": "microsoft/wavlm-large", "emotion2vec": "iic/emotion2vec_base"}
METRICS = ["emo_macroF1", "distress_recall@0.5", "precision@floor", "thr@floor"]

rows = []
agg = {}
for name, seeds in results.items():
    agg[name] = {}
    for k in METRICS:
        vals = np.array([s[k] for s in seeds])
        agg[name][k] = (vals.mean(), vals.std())
        rows.append({"backbone": name, "metric": k,
                     "mean": round(float(vals.mean()), 4), "std": round(float(vals.std()), 4)})
df = pd.DataFrame(rows)
df.to_csv(f"{ART}/results_voice_backbone.csv", index=False)
print(df.to_string(index=False))

# paired per-seed delta emotion2vec - wavlm (the real verdict), when both ran.
if "emotion2vec" in results and "wavlm-large" in results:
    print("\nPaired delta (emotion2vec - wavlm-large), per seed then mean+/-std:")
    for k in ["emo_macroF1", "precision@floor"]:
        d = np.array([results["emotion2vec"][i][k] - results["wavlm-large"][i][k]
                      for i in range(len(SEEDS))])
        print(f"  {k}: {d.mean():+.4f} +/- {d.std():.4f}  (per-seed {np.round(d,4).tolist()})")

# pick the winning backbone by mean emotion macro-F1.
winner = max(agg, key=lambda n: agg[n]["emo_macroF1"][0])
print(f"\nWINNER by emo_macroF1: {winner}")

# save a serving bundle per backbone so the FastAPI app can load either.
for name, (emo_h, saf_h, thr) in artifacts.items():
    d = f"{ART}/artifact_{name}"
    os.makedirs(d, exist_ok=True)
    torch.save(emo_h.state_dict(), f"{d}/emotion_head.pt")
    torch.save(saf_h.state_dict(), f"{d}/safety_head.pt")
    cfg = {"backbone_hf_id": HF_ID[name], "backbone": name, "embed_dim": BACKBONES[name]["dim"],
           "emotions": EMOTIONS, "distress_emotions": sorted(DISTRESS),
           "safety_threshold": float(thr), "recall_floor": RECALL_FLOOR,
           "sample_rate": SR, "max_samples": MAX_SAMPLES, "pooling": "mean",
           "head_dim": HEAD_DIM, "safety_head_dim": 64,
           "is_winner": name == winner}
    with open(f"{d}/config.json", "w") as f:
        json.dump(cfg, f, indent=2)
    print(f"  saved bundle -> {d} (winner={name == winner})")

with open(f"{ART}/results_voice_backbone.json", "w") as f:
    json.dump({"agg": {n: {k: list(v) for k, v in m.items()} for n, m in agg.items()},
               "per_seed": results, "winner": winner, "seeds": SEEDS}, f, indent=2)
print("\nartifacts in /kaggle/working: results_voice_backbone.{csv,json}, sample_val.wav, artifact_*/")
